<a href="https://colab.research.google.com/github/sukhadaghewari/wacv2027_ri_benchmark/blob/main/notebooks/07a_run_claude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Claude models on a case file (text, or text + picture)

Sends every case of one case file to a Claude model (Haiku 5.5, Sonnet 5.5, …) and saves the answers, one line per case: `{"case_id": ..., "response": ...}`, where `response` is the full API reply (answer text, token counts, stop reason). Works for any case file: RI (development or test) and the storm-identification probe (development or test).

- **Exactly the case's prompt.** The user message is the case's `prompt` text, unchanged, with no system prompt, as for the GPT models. Cases from the `..._text_image` case files have an `image` field: their combined picture (notebook 06) follows the text, as image bytes only, never its file name. The order (prompt text, then picture) is the same as for the GPT models.
- **Each case is its own call**, with no memory of other cases. No tools are switched on, so the model can't search the web.
- **Settings are fixed and recorded.** Model, effort and maximum output are saved next to the results (`..._settings.json`), with the batch IDs.
- **Batches at half price.** All cases go through the Message Batches API (usually done within an hour). Submitting and collecting are separate cells, so you can close the notebook and collect later. Cases already in the results file are skipped, so a re-run only sends what is missing or failed.

Try one case with `ask()` first: it costs a fraction of a cent and shows the answer is parsed correctly.

## 1. Setup

Works out the repo folder and makes `src/` importable, as in the other notebooks. In Colab it first clones this repo. It also installs the `anthropic` package if it is missing.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "wacv2027_ri_benchmark"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))
try:
    import anthropic  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)

## 2. Settings

**Case file:** any file in `model_inputs/` (the default is the development-set identification probe without location). For RI with pictures, use `model_inputs/ri_prediction/<dev or test>/..._text_image.jsonl.gz`.

**Model settings:** set `MODEL` to any Claude model ID (see the table in the notes). The current models always use adaptive thinking. `EFFORT = None` leaves each model at its own default effort (Haiku 5.5: `medium`, Sonnet 5.5: `high`), as the GPT models ran at their default reasoning; set `"low"`, `"medium"` or `"high"` to fix it instead. Temperature can't be set on these models. `MAX_TOKENS` caps thinking plus answer; only tokens actually produced are billed.

**Where results go:** `results/<task>/<dev or test>/claude/` in the repo, named like the existing results, for example `results/storm_identification/dev/claude-haiku-5-5_detailed_results_storm_identification_dev_2016_text_only_no_location.jsonl.gz` (committed), with a working `.jsonl` next to it while the run is in progress (not in git). In Colab, set `USE_DRIVE = True` to save them to Google Drive instead (Colab files vanish when the runtime stops); copy them into `results/` before committing. With `USE_DRIVE = True` the pictures are also read from Drive.

**API key:** the Colab secret `ANTHROPIC_API_KEY` if there is one, otherwise the `ANTHROPIC_API_KEY` environment variable, otherwise you are asked to paste it (it is never saved).

In [ ]:
import base64
import gzip
import json
import os
import time
from datetime import datetime, timezone
from getpass import getpass

import anthropic
from prompts import parse_identify_response, parse_response, read_records

# ---- what to send -------------------------------------------------------------------
CASE_FILE = ROOT / "model_inputs" / "storm_identification" / "dev" / "storm_identification_dev_2016_text_only_no_location.jsonl.gz"
LIMIT = None               # e.g. 5 to send only the first 5 cases

# ---- model ----------------------------------------------------------------------------
MODEL = "claude-haiku-5-5"  # e.g. "claude-haiku-5-5", "claude-sonnet-5-5"
EFFORT = None              # None: the model's default effort; or "low" | "medium" | "high"
MAX_TOKENS = 32000         # thinking + answer

# ---- where things are -----------------------------------------------------------------
USE_DRIVE = False          # True: save results to, and read pictures from, Google Drive (Colab)
DRIVE_DIR = "wacv2027_output"   # folder inside MyDrive, as in notebooks 04 and 06

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    images_root = Path("/content/drive/MyDrive") / DRIVE_DIR
    results_root = images_root / "results"
else:
    images_root = ROOT / "model_inputs"
    results_root = ROOT / "results"

# Results file: named like the existing results for this kind of case file.
stem = CASE_FILE.name.removesuffix(".jsonl.gz")
task = CASE_FILE.parent.parent.name                   # model_inputs/<task>/<dev or test>/<case file>
if task == "storm_identification":
    name = f"{MODEL}_detailed_results_{stem}"
    parse = parse_identify_response
else:                                                 # ri_prediction
    name = f"{MODEL}_{stem}"
    parse = parse_response
cases = read_records(CASE_FILE)
results_dir = results_root / task / cases[0]["split"] / "claude"    # e.g. results/storm_identification/dev/claude/
results_path = results_dir / f"{name}.jsonl"                     # working file, appended as answers arrive (not in git)
results_gz = results_dir / f"{name}.jsonl.gz"                    # compressed copy for git and notebooks 08-09
settings_path = results_dir / f"{name}_settings.json"
results_dir.mkdir(parents=True, exist_ok=True)
if results_gz.exists():                                         # continue from the committed answers (e.g. a fresh clone)
    committed = gzip.decompress(results_gz.read_bytes())
    if not results_path.exists() or committed.count(b"\n") > results_path.read_bytes().count(b"\n"):
        results_path.write_bytes(committed)                     # the committed copy has more answers than the working file

if LIMIT:
    cases = cases[:LIMIT]
WITH_IMAGES = "image" in cases[0]                     # the image case files name each case's picture

# ---- API key ----------------------------------------------------------------------------
if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
client = anthropic.Anthropic()

print(f"{len(cases)} cases from {CASE_FILE.name}{' (with pictures)' if WITH_IMAGES else ''}")
print(f"Results: {results_path}")

## 3. Building a request

`build_params` makes the request for one case: the case's prompt text and then its picture (if the case has one), as one user message. No system prompt, no tools and no caching.


In [ ]:
def image_block(path):
    """An image content block from a local PNG: the bytes only, so the file name never reaches the model."""
    data = base64.standard_b64encode(Path(path).read_bytes()).decode()
    return {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": data}}


def build_params(case):
    """The request body for one case."""
    content = [{"type": "text", "text": case["prompt"]}]
    if WITH_IMAGES:
        picture = images_root / case["image"]
        if not picture.exists():
            raise FileNotFoundError(f"no combined picture for {case['case_id']}: run 06_combine_satellite_images.ipynb")
        content.append(image_block(picture))
    params = {"model": MODEL, "max_tokens": MAX_TOKENS, "messages": [{"role": "user", "content": content}]}
    if EFFORT:
        params["output_config"] = {"effort": EFFORT}
    return params


def answer_text(message):
    """Only the answer text, skipping thinking blocks."""
    return "".join(block.text for block in message.content if block.type == "text")


def ask(case):
    """Send one case now (full price) and return the parsed answer."""
    with client.messages.stream(**build_params(case)) as stream:     # streaming: required for large MAX_TOKENS
        message = stream.get_final_message()
    u = message.usage
    print(f"[stop: {message.stop_reason} | input: {u.input_tokens} | output: {u.output_tokens}]")
    return parse(answer_text(message))


def save_settings(batch_ids, pending_batch_ids):
    settings = {
        "case_file": str(CASE_FILE.relative_to(ROOT)), "model": MODEL, "effort": EFFORT or "model default", "max_tokens": MAX_TOKENS,
        "thinking": "adaptive (model default)", "system_prompt": None, "tools": None, "with_images": WITH_IMAGES,
        "message_order": "prompt text, then picture" if WITH_IMAGES else "prompt text",
        "anthropic_sdk": anthropic.__version__, "batch_ids": batch_ids, "pending_batch_ids": pending_batch_ids,
        "updated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    }
    previous = json.loads(settings_path.read_text()) if settings_path.exists() else {}
    changed = {k: previous[k] for k in settings if k in previous and previous[k] != settings[k]
               and k not in ("batch_ids", "pending_batch_ids", "updated_utc", "anthropic_sdk", "openai_sdk")}
    if changed:                                     # answers already in the file were made with these earlier values
        settings["earlier_settings"] = previous.get("earlier_settings", []) + [changed]
    settings_path.write_text(json.dumps({**previous, **settings}, indent=2) + "\n")

## 4. Try one case

Sends the first case at full price and prints the parsed answer, next to the true storm (for the identification probe) or label. If this fails, fix it before sending the batch: a malformed request only shows up after a batch has run.

In [ ]:
case = cases[0]
print(ask(case))
print("truth:", {k: case[k] for k in ["name", "season", "basin", "ri_tplus24", "usa_wind_tplus24h"] if k in case})

## 5. Submit the batch

Sends every case that is not yet in the results file. It refuses to run while an earlier submission hasn't been collected (section 6), so the same cases are never paid for twice. Requests are split into batches of at most 100,000 requests and about 200 MB (the API limit is 256 MB; pictures make requests large). The batch IDs are saved in the settings file, so the next cell can collect them even after a disconnect. Run this once, then run the next cell.

In [ ]:
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

MAX_BATCH_BYTES = 200_000_000
MAX_BATCH_REQUESTS = 100_000

previous = json.loads(settings_path.read_text()) if settings_path.exists() else {}
if previous.get("pending_batch_ids"):
    raise RuntimeError(f"batches {previous['pending_batch_ids']} were submitted but not collected yet: run section 6 first "
                       "(submitting again would send, and pay for, the same cases twice)")

done = set()
if results_path.exists():
    done = {json.loads(line)["case_id"] for line in results_path.open()}
todo = [c for c in cases if c["case_id"] not in done]
print(f"{len(done)} cases already answered, {len(todo)} to send")

batches, current, size = [], [], 0
for case in todo:
    params = build_params(case)
    request = Request(custom_id=case["case_id"], params=MessageCreateParamsNonStreaming(**params))
    n = len(json.dumps(params))
    if current and (size + n > MAX_BATCH_BYTES or len(current) == MAX_BATCH_REQUESTS):
        batches.append(current); current, size = [], 0
    current.append(request); size += n
if current:
    batches.append(current)

new_ids = []
for requests in batches:
    batch = client.messages.batches.create(requests=requests)
    new_ids.append(batch.id)
    save_settings(previous.get("batch_ids", []) + new_ids, new_ids)     # saved at once, so a later failure loses nothing
    print(f"Submitted {len(requests)} requests as {batch.id}")

## 6. Collect the answers

Waits until every batch has ended, then appends each answer to the working results file (`.jsonl`) as `{"case_id": ..., "response": ...}` and writes the compressed copy (`.jsonl.gz`), which is the file to commit. Cases that errored or expired are listed and not saved; run section 5 again to resend just those.

In [ ]:
settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
done = {json.loads(line)["case_id"] for line in results_path.open()} if results_path.exists() else set()

saved, failed = 0, []
for batch_id in settings.get("pending_batch_ids", []):
    while (batch := client.messages.batches.retrieve(batch_id)).processing_status != "ended":
        print(f"{batch_id}: {batch.request_counts.processing} still processing...")
        time.sleep(60)
    with results_path.open("a") as out:
        for entry in client.messages.batches.results(batch_id):
            if entry.custom_id in done:
                continue
            if entry.result.type == "succeeded":
                out.write(json.dumps({"case_id": entry.custom_id, "response": entry.result.message.model_dump(mode="json")}) + "\n")
                done.add(entry.custom_id); saved += 1
            else:
                failed.append((entry.custom_id, entry.result.type))

if settings:
    settings["pending_batch_ids"] = []                # collected: section 5 may submit again
    settings_path.write_text(json.dumps(settings, indent=2) + "\n")
print(f"Saved {saved} new answers; {len(done)} of {len(cases)} cases answered in total")
if failed:
    print(f"{len(failed)} not answered (run section 5 again to resend):", failed[:10])

# Compressed copy of all answers: this is the file to commit (plain .jsonl files are not in git).
# Not for a LIMIT trial: notebooks 08 and 09 expect a results file to answer every case.
if LIMIT:
    print(f"LIMIT run: answers kept only in {results_path.name}; run without LIMIT to write the file to commit")
else:
    with open(results_gz, "wb") as raw, gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as gz:
        gz.write(results_path.read_bytes())
    print(f"Saved {results_gz}")

## 7. Check the answers

Parses every saved answer with the same parser used for the GPT results, and counts answers that were cut off or unreadable.

In [ ]:
from collections import Counter

rows = [json.loads(line) for line in results_path.open()]
stops = Counter(r["response"]["stop_reason"] for r in rows)
bad = []
for r in rows:
    text = "".join(b["text"] for b in r["response"]["content"] if b["type"] == "text")
    try:
        parse(text)
    except ValueError as err:
        bad.append((r["case_id"], str(err)))
print(f"{len(rows)} answers | stop reasons: {dict(stops)} | unreadable: {len(bad)}")
for case_id, err in bad[:10]:
    print("  ", case_id, err)

## Notes

- **Claude models** (prices per million input / output tokens; the Batch API halves them):

  | Model | `MODEL` | Price | Default effort |
  |---|---|---|---|
  | Claude Haiku 5.5 | `claude-haiku-5-5` | $0.10 / $0.50 | `medium` |
  | Claude Sonnet 5.5 | `claude-sonnet-5-5` | $2 / $10 | `high` |
  | Claude Opus 5.5 | `claude-opus-5-5` | $4 / $20 | `medium` |

- **Knowledge cutoff:** these models' knowledge runs to June 2026, after every storm in both sets, so they may recognise storms. Run the identification probe for each model, as for the GPT models: both versions on the development set, without location on the test set.
- **Tokens:** these models count about 45% more tokens than the OpenAI token counts for the same text; a combined picture adds about 800 tokens.
- **Settings of earlier runs:** the committed Claude Haiku 5.5 results were made with effort `medium` and `MAX_TOKENS` 16,000 (see each `_settings.json`). If you resend cases into an existing results file with different settings, the old values are kept under `earlier_settings`.
- **Never send file names:** pictures are sent as bytes; the `case_id` goes only in `custom_id`, which the model never sees.